In [3]:
import kagglehub
path = kagglehub.dataset_download("migeruj/videogames-predictive-model")

100%|██████████| 71.6k/71.6k [00:00<00:00, 42.2MB/s]

Extracting files...


In [4]:
import pandas as pd
import os

df = pd.read_csv(path + '/dato.csv')

print(df.shape)
print("-----")
print(df.head())
print("-----")
print(df.info())
print("-----")
print(df.isnull().sum())
print("-----")

(7112, 10)
-----
  Platform     Genre Publisher NA_Sales EU_Sales JP_Sales Other_Sales  \
0      Wii    Sports  Nintendo    41,36    28,96     3,77        8,45   
1      Wii    Racing  Nintendo    15,68     12,8     3,79        3,29   
2      Wii    Sports  Nintendo    15,61    10,95     3,28        2,95   
3       DS  Platform  Nintendo    11,28     9,15      6,5        2,88   
4      Wii      Misc  Nintendo    13,96     9,18     2,93        2,84   

  Global_Sales Rating Critic_Score_Class  
0        82,54      E              Bueno  
1        35,57      E          Excelente  
2        32,78      E          Excelente  
3        29,81      E          Excelente  
4        28,92      E               Malo  
-----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7112 entries, 0 to 7111
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Platform            7112 non-null   object
 1   Genre               7112

In [ ]:
columnas_ventas = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

for col in columnas_ventas:
    df[col] = df[col].str.replace(',', '.').astype(float)

# Verificar que funcionó
print(df.dtypes)
print('---')
print(df.describe())

Platform               object
Genre                  object
Publisher              object
NA_Sales              float64
EU_Sales              float64
JP_Sales              float64
Other_Sales           float64
Global_Sales          float64
Rating                 object
Critic_Score_Class     object
dtype: object
---
          NA_Sales     EU_Sales     JP_Sales  Other_Sales  Global_Sales
count  7112.000000  7112.000000  7112.000000  7112.000000   7112.000000
mean      0.388567     0.232537     0.062652     0.081347      0.765307
std       0.953982     0.680028     0.283475     0.265864      1.936692
min       0.000000     0.000000     0.000000     0.000000      0.010000
25%       0.060000     0.020000     0.000000     0.010000      0.110000
50%       0.150000     0.060000     0.000000     0.020000      0.290000
75%       0.390000     0.202500     0.010000     0.070000      0.742500
max      41.360000    28.960000     6.500000    10.570000     82.540000


In [ ]:
# 1. Ventas globales por género
print("VENTAS POR GÉNERO:")
print(df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False))
print('---')

# 2. Top 10 publishers por ventas globales
print("TOP 10 PUBLISHERS:")
print(df.groupby('Publisher')['Global_Sales'].sum().sort_values(ascending=False).head(10))
print('---')

# 3. Ventas por plataforma
print("VENTAS POR PLATAFORMA:")
print(df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10))

VENTAS POR GÉNERO:
Genre
Action          1239.48
Sports           860.27
Shooter          840.84
Role-Playing     511.81
Racing           482.26
Misc             426.73
Platform         383.21
Fighting         253.91
Simulation       210.88
Adventure         81.88
Puzzle            79.87
Strategy          71.72
Name: Global_Sales, dtype: float64
---
TOP 10 PUBLISHERS:
Publisher
Electronic Arts                 898.12
Nintendo                        862.30
Activision                      549.01
Sony Computer Entertainment     395.52
Take-Two Interactive            355.73
Ubisoft                         348.19
Microsoft Game Studios          219.25
THQ                             164.64
Sega                            151.56
Konami Digital Entertainment    142.29
Name: Global_Sales, dtype: float64
---
VENTAS POR PLATAFORMA:
Platform
PS2     965.91
X360    868.48
PS3     793.33
Wii     676.39
DS      385.21
PS4     264.91
X       217.10
PS      210.81
PC      195.51
PSP     190.97
Name: Gl

In [10]:
#Ver duplicados
print("Duplicados: ", df.duplicated().sum())
print("----")

#Ver valores únicos de columnas categóricas
print("Géneros: ", df['Genre'].unique())
print("Ratings: ", df['Rating'].unique())
print("Critic_Score_Class: ", df['Critic_Score_Class'].unique())

Duplicados:  25
----
Géneros:  ['Sports' 'Racing' 'Platform' 'Misc' 'Action' 'Puzzle' 'Shooter'
 'Fighting' 'Simulation' 'Role-Playing' 'Adventure' 'Strategy']
Ratings:  ['E' 'M' 'T' 'E10+' 'AO' 'RP']
Critic_Score_Class:  ['Bueno' 'Excelente' 'Malo' 'Aceptable']
Rating
T       3568
M       2127
E10+    1403
E         10
AO         2
RP         2
Name: count, dtype: int64


In [9]:
# Unificar K-A con E ya que son lo mismo
df['Rating'] = df['Rating'].replace(['K-A','E'])

# Verificar cuantos juegos hay por rating
print(df['Rating'].value_counts())
print("---")
# Ver distribuición de Critic_Score_Class
print(df['Critic_Score_Class'].value_counts())


Rating
T       3568
M       2127
E10+    1403
E         10
AO         2
RP         2
Name: count, dtype: int64
Rating
T       3568
M       2127
E10+    1403
E         10
AO         2
RP         2
Name: count, dtype: int64
---
Critic_Score_Class
Excelente    1997
Bueno        1957
Aceptable    1698
Malo         1460
Name: count, dtype: int64


/tmp/ipykernel_1601/4193355362.py:3: FutureWarning: Series.replace without 'value' and with non-dict-like 'to_replace' is deprecated and will raise in a future version. Explicitly specify the new values instead.
  df['Rating'] = df['Rating'].replace(['K-A','E'])


In [12]:
!pip install pandasql
from pandasql import sqldf

# Función auxiliar para simplificar las consultas
pysql = lambda q: sqldf(q, globals())

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=77546b32e79c79efb340f2c491b016021f2df8e822bbd9302d902bd52e5fb0c1
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [13]:
# ¿Qué género vende más en cada región?
query1 = """
SELECT Genre,
    SUM(NA_Sales) as Ventas_NA,
    SUM(EU_Sales) as Ventas_EU,
    SUM(JP_Sales) as Ventas_JP
FROM df
GROUP BY Genre
ORDER BY Ventas_NA DESC
"""
print(pysql(query1))


           Genre  Ventas_NA  Ventas_EU  Ventas_JP
0        Shooter      238.0      106.0        1.0
1         Action      233.0      139.0        9.0
2         Sports      205.0      122.0       16.0
3           Misc      120.0       59.0       15.0
4         Racing      102.0       65.0       15.0
5       Platform       92.0       43.0       22.0
6   Role-Playing       78.0       40.0       45.0
7       Fighting       41.0       10.0        5.0
8     Simulation       32.0       25.0       13.0
9         Puzzle       11.0       12.0        7.0
10     Adventure        7.0        5.0        1.0
11      Strategy        5.0        3.0        0.0


In [14]:
# ¿Los juegos mejor valorados por la crítica venden más?
query2 = """
SELECT
    Critic_Score_Class,
    COUNT(*) as Num_Juegos,
    ROUND(AVG(Global_Sales), 2) as Ventas_Promedio,
    ROUND(SUM(Global_Sales), 2) as Ventas_Totales
FROM df
GROUP BY Critic_Score_Class
ORDER BY Ventas_Promedio DESC
"""
print(pysql(query2))

  Critic_Score_Class  Num_Juegos  Ventas_Promedio  Ventas_Totales
0          Excelente        1997             1.11          2220.0
1              Bueno        1957             0.34           660.0
2          Aceptable        1698             0.15           257.0
3               Malo        1460             0.08           119.0


In [18]:
# ¿Qué plataforma tiene mejor rendimiento por juego?
query3 = """
SELECT
    Platform,
    COUNT(*) as Num_Juegos,
    ROUND(AVG(Global_Sales), 2) as Ventas_Promedio
FROM df
GROUP BY Platform
ORDER BY Ventas_Promedio DESC
LIMIT 10
"""
print(pysql(query3))

  Platform  Num_Juegos  Ventas_Promedio
0      Wii         493             1.02
1       PS         154             0.97
2      PS4         255             0.73
3     X360         888             0.62
4      PS3         790             0.61
5       DS         472             0.54
6     XOne         169             0.50
7      3DS         161             0.50
8      PS2        1169             0.47
9     WiiU          89             0.45


In [19]:
#¿Qué publisher domina en cada región?
query4 = """
SELECT
    Publisher,
    ROUND(SUM(NA_Sales), 2) as Ventas_NA,
    ROUND(SUM(EU_Sales), 2) as Ventas_EU,
    ROUND(SUM(JP_Sales), 2) as Ventas_JP,
    ROUND(SUM(Global_Sales), 2) as Ventas_Global
FROM df
GROUP BY Publisher
ORDER BY Ventas_Global DESC
LIMIT 10
"""
print(pysql(query4))

                                Publisher  Ventas_NA  Ventas_EU  Ventas_JP  \
0                                Nintendo      259.0      179.0       92.0   
1                         Electronic Arts      169.0      111.0        3.0   
2                              Activision      159.0       79.0        0.0   
3             Sony Computer Entertainment       94.0       57.0       16.0   
4                    Take-Two Interactive      107.0       52.0        0.0   
5                                 Ubisoft       64.0       44.0        0.0   
6                  Microsoft Game Studios       95.0       23.0        0.0   
7            Konami Digital Entertainment       13.0       13.0        4.0   
8  Warner Bros. Interactive Entertainment       23.0        6.0        0.0   
9                                    Sega       15.0        7.0        0.0   

   Ventas_Global  
0          731.0  
1          515.0  
2          358.0  
3          269.0  
4          257.0  
5          196.0  
6       